# Tanka 02: jsonnet-bundler and k8s-libsonnet

`jb` vendors Jsonnet libraries from git into `vendor/` and pins them in `jsonnetfile.lock.json`.
`k8s-libsonnet` is generated from the Kubernetes OpenAPI spec, one directory per version, so the
API surface of `k.apps.v1.deployment` is exactly what the cluster accepts.


In [ ]:
cd /source/work/tanka-lab
export HOME=/tmp
jb install github.com/jsonnet-libs/k8s-libsonnet/1.33@main >/dev/null 2>&1 && jq '.dependencies[] | .source.git.subdir + " " + .version' jsonnetfile.lock.json


In [ ]:
cd /source/work/tanka-lab
ls vendor | head; echo; ls vendor/github.com/jsonnet-libs/k8s-libsonnet/ ; echo; ls vendor/1.33/_gen/apps/v1 | head -5


In [ ]:
cd /source/work/tanka-lab
grep -n "deployment" vendor/1.33/_gen/apps/v1/main.libsonnet | head -3; grep -n "new(" vendor/1.33/_gen/apps/v1/deployment.libsonnet | head -3


Switch the project to 1.33: the `k.libsonnet` shim is the single place that decides which API version your code compiles against.


In [ ]:
cd /source/work/tanka-lab
sed -i 's#k8s-libsonnet/1.32/main.libsonnet#k8s-libsonnet/1.33/main.libsonnet#' lib/k.libsonnet && cat lib/k.libsonnet && tk show environments/default --dangerous-allow-redirect | grep -c '^kind:'


In [ ]:
cd /source/work/tanka-lab
jb update >/dev/null 2>&1; jq -r '.dependencies[] | .source.git.subdir + " @ " + .version[0:12]' jsonnetfile.lock.json


Try it: `jb install github.com/grafana/jsonnet-libs/grafana-builder@master` and build a dashboard next to the Deployment. Same language, different API.
